# Paper Citation Counts — C3 / C5 / C10 / C_all

Forward-citation counts received by each publication within four windows (**3, 5, 10, full**),
from the Dimensions reference graph. Primary key: `paper_id` (`pub.…`). Same kernel as
`OpenAlex/notebook/paper_citation.ipynb`.

## Input
```
Dimensions/cache/pub_year_source_map.npz   # publication years
Dimensions/cache/paper_graph.npz           # c_from, c_to, year, uni_mag -- built by dim.build_graph() on first run
```

## Metric
$C_W(P)=\#\{c \text{ cites } P: 0 \le y_c - y_P \le W\}$; $C_{\mathrm{all}}$ counts all citers with $y_c \ge y_P$.

## Output
`Dimensions/output/paper_citation.parquet` — `paper_id, C_3, C_5, C_10, C_all`. Dimensions' own
`citations_count` / `metrics.times_cited` are in `paper_metadata` for comparison; they count
citations from the whole Dimensions index at export time, this counts edges present in the dump.

In [1]:
import os, sys, gc, glob, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Dimensions')
import dim_common as dim
ROOT = dim.BASE; OUT = dim.OUT
print('dump:', dim.ROOT)
OUT_FP = f'{OUT}/paper_citation.parquet'
WINSFX = [(3, '_3'), (5, '_5'), (10, '_10'), (-1, '_all')]
# Everything below is fed by notebook/references_w_year.ipynb: it writes the per-publication
# map (year + source), the scalar and author parts, and the edge table with both years and both
# source ids. Build it once before running this notebook.
assert dim.have_consolidated(), (
    'run notebook/references_w_year.ipynb first -- it builds the map, the scalar parts and the edge table')
dim.summary()

dump: /project/jevans/dimensions/dimensions/dimensions_june_2025
dump     : /project/jevans/dimensions/dimensions/dimensions_june_2025
cache    : /project/jevans/Dawoon/Science of Science/Dimensions/cache
output   : /project/jevans/Dawoon/Science of Science/Dimensions/output
  consolidated edge table: present
  map            2.80 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_year_source_map.npz
  graph        not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_graph.npz
  csr          not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_csr.npz
  journal      not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_journal.parquet
  fos          not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_fos.parquet
  pat2pub      not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/patent2pub_edges.parquet
  scalars       4219 parts  /project/jevans/Dawoon/Science o

## 1. Load (or build) the citation graph

In [2]:
%%time
c_from, c_to, year, uni_mag = dim.load_graph()   # builds the cache on first run
n = len(year)
print(f'{len(c_from):,} edges, {n:,} publications')

[1/2] 155,441,856 publications with a year, from pub_year_source_map.npz
[2/2] edges from 4219 partitions of /project/jevans/Dawoon/Science of Science/Dimensions/output/references_w_year ...
      500/4219  279,225,956 edges  [1058s]
      1000/4219  543,615,921 edges  [2047s]
      1500/4219  820,126,128 edges  [3071s]
      2000/4219  1,059,990,478 edges  [3963s]
      2500/4219  1,295,978,869 edges  [4838s]
      3000/4219  1,517,744,497 edges  [5653s]
      3500/4219  1,748,272,586 edges  [6508s]
      4000/4219  2,001,853,776 edges  [7455s]
[8024s] 2,141,693,663 edges, 155,441,856 publications -> /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_graph.npz  (19.0 GB)
2,141,693,663 edges, 155,441,856 publications


## 2. Count citers per window (vectorised over edges — kernel identical to OpenAlex)

In [3]:
%%time
# forward citation counts per window, fully VECTORIZED over edges (citer = c_from, cited = c_to).
# window w: count citers q of paper P with 0 <= year[q] - year[P] <= w  (C_all = all with diff >= 0).
y = year.astype(np.int32)
diff = y[c_from] - y[c_to]                 # per-edge: citer_year - cited_year
del y; gc.collect()
valid = diff >= 0
res = {}
res['_all'] = np.bincount(c_to[valid], minlength=n).astype(np.int64)
for w, sfx in [(10, '_10'), (5, '_5'), (3, '_3')]:
    res[sfx] = np.bincount(c_to[valid & (diff <= w)], minlength=n).astype(np.int64)
del diff, valid; gc.collect()
print('citation counts computed (vectorized):', {k: int(v.sum()) for k, v in res.items()})

citation counts computed (vectorized): {'_all': 2138656982, '_10': 1425725632, '_5': 898931125, '_3': 581227562}


## 3. Save + example

In [4]:
out = pd.DataFrame({'paper_id': dim.code_to_id(uni_mag)})
for _, s in WINSFX: out[f'C{s}'] = res[s]
out.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(out):,} rows, {len(out.columns)} cols)')
for _, s in WINSFX: print(f'  C{s}: mean {out[f"C{s}"].mean():.2f}  >0 {(out[f"C{s}"]>0).mean()*100:.1f}%')
display(out.head(10))

WROTE /project/jevans/Dawoon/Science of Science/Dimensions/output/paper_citation.parquet  (155,441,856 rows, 5 cols)
  C_3: mean 3.74  >0 43.0%
  C_5: mean 5.78  >0 47.1%
  C_10: mean 9.17  >0 50.6%
  C_all: mean 13.76  >0 53.9%


,paper_id,C_3,C_5,C_10,C_all
0,pub.1000000001,0,0,0,0
1,pub.1000000002,14,21,34,52
2,pub.1000000003,0,0,0,0
3,pub.1000000004,1,2,4,4
4,pub.1000000005,0,0,0,0
5,pub.1000000006,19,27,45,61
6,pub.1000000007,65,100,196,201
7,pub.1000000008,2,2,7,11
8,pub.1000000009,4,14,26,73
9,pub.1000000010,8,11,24,35
